# RAG Pipeline - Data Ingestion to Vector DB

In [93]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from pathlib import Path
from tqdm import tqdm

In [94]:
# Step 1: Read all pdfs in directory

def process_pdfs(pdf_directory):
    documents = []
    pdf_dir = Path(pdf_directory)

    # find pdfs recursively
    pdf_files = list(pdf_dir.rglob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory} to process.")

    for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
        print(f"Processing {pdf_file}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            doc = loader.load()

            # add source info to metadata
            for d in doc:
                d.metadata["source_file"] = str(pdf_file.name)
                d.metadata["file_type"] = "pdf"

            documents.extend(doc)
            print(f"Loaded {len(doc)} documents from {pdf_file.name}.")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"Finished processing PDFs. Total documents loaded: {len(documents)}.")
    return documents

all_documents = process_pdfs("../data/pdf")

Found 4 PDF files in ../data/pdf to process.


Processing PDFs:  25%|██▌       | 1/4 [00:00<00:00,  9.98it/s]

Processing ../data/pdf/ml-models.pdf...
Loaded 32 documents from ml-models.pdf.
Processing ../data/pdf/The Ultimate Python Handbook.pdf...


Processing PDFs: 100%|██████████| 4/4 [00:00<00:00, 14.20it/s]

Loaded 61 documents from The Ultimate Python Handbook.pdf.
Processing ../data/pdf/python-notes.pdf...
Loaded 10 documents from python-notes.pdf.
Processing ../data/pdf/ml-notes.pdf...
Loaded 11 documents from ml-notes.pdf.
Finished processing PDFs. Total documents loaded: 114.


In [95]:
all_documents

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260531232148', 'source': '../data/pdf/ml-models.pdf', 'file_path': '../data/pdf/ml-models.pdf', 'total_pages': 32, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260531232148', 'page': 0, 'source_file': 'ml-models.pdf', 'file_type': 'pdf'}, page_content='18 \n \n• \nAdd to G all minimal specializations h of g such that \n• \nh is consistent with d, and some member of S is more specific than h \n• \nRemove from G any hypothesis that is less general than another hypothesis in G \nCANDIDATE- ELIMINTION algorithm using version spaces \n \n1.7.4 An Illustrative Example \n \n \nExample \nSky \nAirTemp \nHumidity \nWind \nWater \nForecast \nEnjoySport \n1 \nSunny \nWarm \nNormal \nStrong \nWarm \nSame \nYes \n2 \nSunny \nWarm \nHigh \nStrong \nWarm \nSame \nYes \n3 \nRainy \nCold \nHigh \nStrong \nWarm \nChange \n

In [96]:
# Step 2: Text splitting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    # split documents into smaller chunks for better RAG performance

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # maximum size of each chunk in characters
        chunk_overlap=chunk_overlap, # amount of character overlap between chunks to maintain context
        length_function=len, # function to calculate the length of text (default is len)
        separators=["\n\n", "\n", " ", ""] # list of separators to use for splitting text, ordered by priority (default is ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}") # print first 200 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}") # print metadata of the first chunk
    return split_docs



In [97]:
chunks = split_documents(all_documents)
chunks

Split 114 documents into 237 chunks.

Example chunk:
Content: 18 
 
• 
Add to G all minimal specializations h of g such that 
• 
h is consistent with d, and some member of S is more specific than h 
• 
Remove from G any hypothesis that is less general than anoth
Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260531232148', 'source': '../data/pdf/ml-models.pdf', 'file_path': '../data/pdf/ml-models.pdf', 'total_pages': 32, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260531232148', 'page': 0, 'source_file': 'ml-models.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260531232148', 'source': '../data/pdf/ml-models.pdf', 'file_path': '../data/pdf/ml-models.pdf', 'total_pages': 32, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260531232148', 'page': 0, 'source_file': 'ml-models.pdf', 'file_type': 'pdf'}, page_content='18 \n \n• \nAdd to G all minimal specializations h of g such that \n• \nh is consistent with d, and some member of S is more specific than h \n• \nRemove from G any hypothesis that is less general than another hypothesis in G \nCANDIDATE- ELIMINTION algorithm using version spaces \n \n1.7.4 An Illustrative Example \n \n \nExample \nSky \nAirTemp \nHumidity \nWind \nWater \nForecast \nEnjoySport \n1 \nSunny \nWarm \nNormal \nStrong \nWarm \nSame \nYes \n2 \nSunny \nWarm \nHigh \nStrong \nWarm \nSame \nYes \n3 \nRainy \nCold \nHigh \nStrong \nWarm \nChange \n

# Embedding and Vector store db

In [98]:
import numpy as np
from sentence_transformers import SentenceTransformer # for generating vector embeddings from text chunks
import chromadb
from chromadb.config import Settings
import uuid # every record passed into vector db will have a unique id.
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity # for calculating cosine similarity between query and document vectors

In [99]:
# Step 3: Chunks to Vector embeddings.
class EmbeddingManager:
    # Handles document embedding generation using SentenceTransformer models.

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        '''
        Initialize the embedding manager.

        Args:
            model_name (str): The name of the SentenceTransformer model to use for generating embeddings.
        '''
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        '''
        Load the SentenceTransformer model specified by model_name.
        '''
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Emdedding dimension is: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error occurred while loading the model ({self.model_name}): {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        '''
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): A list of text strings to generate embeddings for.

        Returns:
            np.ndarray: A 2D array where each row corresponds to the embedding of a text with shape (len(texts), self.model.get_sentence_embedding_dimension())
        '''
        if not self.model:
            raise ValueError("Embedding model is not loaded. Please check the model name and try again.")

        print(f"Generating embeddings for {len(texts)} texts...")
        emebeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully with shape: {emebeddings.shape}.")
        return emebeddings

# initialize the embedding manager with the desired model
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9434.05it/s]


Model loaded successfully. Emdedding dimension is: 384


/var/folders/fh/4w0rpl2s23g9tjfk1ctgznlc0000gn/T/ipykernel_6664/3335900921.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Emdedding dimension is: {self.model.get_sentence_embedding_dimension()}")


# Vector Store

In [100]:
class VectorStore:
    '''
    Manages document embeddings in a chromadb vector store database.
    '''

    def __init__(self, collection_name: str =  "pdf_documents", persist_directory: str = "../data/vector_store"):
        '''
        Initialize the vector store.

        Args:
            collection_name (str): The name of the chromadb collection.
            persist_directory (str): The directory path where the vector store database will be persisted on disk.
        '''

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        '''
        Initialize chromadb client and collection.'''

        try:
            # Create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG."}
            )

            print(f"Vector store initialized. Collection name: '{self.collection_name}', Persist directory: '{self.persist_directory}'")
            print(f"Existing documents in collection:  {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        '''
        Add documents and their corresponding embeddings to the vector store.

        Args:
            documents (List[Any]): A list of document objects (e.g., Langchain Document instances) to be added to the vector store.
            embeddings (np.ndarray): A 2D array of shape (len(documents), embedding_dimension) containing the embeddings corresponding to each document.
        '''
        if not self.collection:
            raise ValueError("Vector store collection is not initialized. Please check the initialization and try again.")

        if len(documents) != embeddings.shape[0]:
            raise ValueError(f"The number of documents ({len(documents)}) must match the number of embeddings ({embeddings.shape[0]}).")

        print(f"Adding {len(documents)} documents to the vector store...")

        # prepare data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # ? generate a unique id for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # ? prepare metadata.
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # ? document content
            documents_text.append(doc.page_content)

            # ? embedding
            embeddings_list.append(embedding.tolist())
        
        # ? add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store. Total documents in collection: {self.collection.count()}.")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector store initialized. Collection name: 'pdf_documents', Persist directory: '../data/vector_store'
Existing documents in collection:  86


In [101]:

# ? Convert text to embeddings
texts = [doc.page_content for doc in chunks]

# ? Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# ? store in the vector store db
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 237 texts...


Batches: 100%|██████████| 8/8 [00:00<00:00,  8.83it/s]

Embeddings generated successfully with shape: (237, 384).
Adding 237 documents to the vector store...
Successfully added 237 documents to the vector store. Total documents in collection: 323.


# RAG Retriever Pipeline from vector store

In [102]:
class RAGRetriever:
    '''
    Handles query based retrieval from vector store using cosine similarity.
    '''

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        '''
        Initialize the RAG retriever.

        Args:
            vector_store (VectorStore): vector store containing document embeddings.
            embedding_manager (EmbeddingManager): embedding manager to generate query embeddings.
        '''

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager  
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        '''
        retrieve relevant documents from vector store based on query.

        Args:
            query (str): The input query string for which relevant documents need to be retrieved.
            top_k (int): The number of top relevant documents to retrieve based on cosine similarity scores. Default is 5.
            score_threshold (float): Minimum cosine similarity score required for a document to be considered relevant. Default is 0.0 (no threshold).
        
        Returns:
            list of dicts containing retrieved documents and their metadata, sorted by relevance score in descending order.
        '''

        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}...")

        # ? generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # ? search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # ? process results 
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # ? calculate cosine similarity score between query embedding and document embedding (chromadb uses cosine)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} relevant documents for the query.")

            else:   
                print("No documents found in vector store for the query.")
                return []
            
            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving documents from vector store: {e}")
            raise

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

In [103]:
rag_retriever

In [110]:
# using the above retriever to get relevant documents for a query
# rag_retriever.retrieve("What is python?")
# rag_retriever.retrieve("What is python?", top_k=2, score_threshold=0.5)
rag_retriever.retrieve("strings in python", top_k=2, score_threshold=0.1)

Retrieving documents for query: 'strings in python' with top_k=2 and score_threshold=0.1...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.34it/s]

Embeddings generated successfully with shape: (1, 384).
Retrieved 2 relevant documents for the query.


[{'id': 'doc_78e4d6cc_127',
  'content': '13 \n \nCHAPTER 3 – STRINGS \nString is a data type in python. \nString is a sequence of characters enclosed in quotes. \nWe can primarily write a string in these three ways. \na =\'harry\'       # Single quoted string \nb = "harry"      # Double quoted string \nc = \'\'\'harry\'\'\'  # Triple quoted string \nSTRING SLICING \nA string in python can be sliced for getting a part of the strings. \nConsider the following string: \n  \n \n \nThe index in a sting starts from 0 to (length -1) in Python. In order to slice a string, we use \nthe following syntax:  \n  \n \n \nNegative Indices: Negative indices can also be used as shown in the figure above. -1 \ncorresponds to the (length - 1) index, -2 to (length - 2).',
  'metadata': {'author': 'Harry Codes',
   'keywords': '',
   'doc_index': 127,
   'creationdate': '2024-06-16T02:12:11+05:30',
   'total_pages': 61,
   'subject': '',
   'title': '',
   'file_type': 'pdf',
   'creationDate': "D:2024061

# Integration of vector db  context pipeline with LLM output

In [111]:
# Simple RAG pipeline with a LOCAL llama.cpp LLM (OpenAI-compatible endpoint)
# Start the server first (see local-inference-setup.md):
#   llama-server -hf Qwen/Qwen2.5-1.5B-Instruct-GGUF:Q4_K_M --port 8080 -ngl 99 -c 8192 --parallel 4
from langchain_openai import ChatOpenAI

# ? Point LangChain at the local llama.cpp server's OpenAI-compatible API.
#   llama.cpp doesn't authenticate, but the client needs a non-empty api_key.
#   `model` is just a label here; llama.cpp serves whatever model was loaded at launch.
llm = ChatOpenAI(
    base_url="http://localhost:8080/v1",
    api_key="sk-local",
    model="qwen2.5-1.5b-instruct-q4_k_m",
    temperature=0.1,
    max_tokens=1024,
)

# ? 2. Simple RAG Function: retrieve context + generate response
def rag_simple(query,retriever, llm, top_k=3):
    # ? retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        print("No relevant documents found for the query. Generating response without context.")

    # ? Generate the response
    prompt = f'''Use the following context to answer the question carefully.
    Context: {context}

    Question: {query}

    Answer:'''

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [117]:
answer = rag_simple("what are trees in machine learning?", rag_retriever, llm)
print(f"Answer: {answer}")

Retrieving documents for query: 'what are trees in machine learning?' with top_k=3 and score_threshold=0.0...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.33it/s]

Embeddings generated successfully with shape: (1, 384).
Retrieved 3 relevant documents for the query.


Answer: Trees in machine learning are a type of supervised learning algorithm used for classification and regression tasks. They are used to predict a target variable based on one or more predictor variables. The tree is a tree-like model of decisions and their possible consequences, including the chance of mutual recursion. The tree is built in a way that each internal node represents a feature (or variable) and each leaf node represents a class label (for classification) or a continuous value (for regression). The tree is built by recursively partitioning the data based on the feature that results in the most significant reduction in the variance of the target variable.


# Enhanced RAG Pipeline features : Part 1

In [123]:

def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    '''
    RAG Pipeline with extra features.

    Returns: answers, sources, confidence score, and optionally the retrieved context.
    '''

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            'answer': "No relevant documents found to answer the query.",
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # ? Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + "..." # first 300 chars as preview
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # ? Generate answer
    prompt = f"Use the following context to answer the question carefully.\nContext: {context}\nQuestion: {query}\nAnswer:"
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    
    return output


In [125]:
results = rag_advance("how do trees work in machine learning?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", results['answer'])
print("Sources:", results['sources'])
print("Confidence:", results['confidence'])
print("Context preview:", results['context'][:300])


Retrieving documents for query: 'how do trees work in machine learning?' with top_k=3 and score_threshold=0.1...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.71it/s]

Embeddings generated successfully with shape: (1, 384).
Retrieved 3 relevant documents for the query.


Answer: Trees in machine learning work by recursively splitting the data into subsets based on certain criteria or features. The process starts with an initial "root" node, which represents the entire dataset. The tree then splits the data at each node based on the feature that best separates the data into distinct groups. This process continues until the tree reaches a stopping criterion, such as a maximum depth or a minimum number of samples required in a leaf node.

At each node, the data is split based on a specific feature, and the resulting subsets are further split recursively until the tree is fully grown. The final outcome or "leaf" nodes represent the decisions or the final outcomes for the data points. The tree is then used to make predictions for new, unseen data by traversing the tree from the root to a leaf node based on the values of the features in the new data point.

The key advantage of decision trees is their ability to handle both numerical and categorical data, an

# Enhanced RAG Pipeline features : Part 2

In [126]:

# ? Streaming, Citations, History, Summarization.
from typing import List, Dict, Any
import time

In [127]:
class AdvancedRAGPipeline:
    def __init__(self, retriever: RAGRetriever, llm: ChatOpenAI):
        self.retriever = retriever
        self.llm = llm
        self.history = [] # store query history

    def query(self, question:str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # ? similar to rag_advance but also stores history and can be extended with more features like streaming, citations formatting, etc.

        # ? retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant documents found to answer the query.",
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:300] + "..." # first 300 chars
            } for doc in results]

            # ? streaming answer simulation
            prompt = f"Use the following context to answer the question carefully.\nContext: {context}\nQuestion: {question}\nAnswer:"
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)

                print()

            respose = self.llm.invoke([prompt.format(context=context, query=question)])
            answer = respose.content

            # ? add citations to answer
            citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
            answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

            # ? Optionally summarize the answer
            summary = None
            if summarize and answer:
                summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
                summary_resp = self.llm.invoke([summary_prompt])
                summary = summary_resp.content

            # ? Store query history
            self.history.append({
                'question': question,
                'answer': answer,
                'sources': sources,
                'summary': summary
            })

            return {
                'question': question,
                'answer': answer_with_citations,
                'sources': sources,
                'summary': summary,
                'history': self.history
            }

In [129]:
advanced_rag = AdvancedRAGPipeline(retriever=rag_retriever, llm=llm)
result = advanced_rag.query("how do decision trees work in machine learning?", top_k=3, min_score=0.1, stream=True, summarize=True)

print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'how do decision trees work in machine learning?' with top_k=3 and score_threshold=0.1...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.31it/s]

Embeddings generated successfully with shape: (1, 384).
Retrieved 3 relevant documents for the query.
Streaming answer:
Use the following context to answer the question carefully.
Context: the input is and what the corresponding output is in the training data) where the data is continuously 
split according to a certain parameter. The tree can be explained b

y two entities, namely decision 
nodes and leaves. The leaves are the decisions or the final outcomes. And the decision nodes are where 
the data is split. 
 
An example of a decision tree can be explained using above binary tree. Let’s say you want to predict 
whether a person is fit given their information like age, eating habit, and physical activity, etc. The

the input is and what the corresponding output is in the training data) where the data is continuously 
split according to a certain parameter. The tree can be explained by two entities, namely decision 
nodes and leaves. The leaves are the decisions or the final outcomes. And the decision nodes are where 
the data is split. 
 
An example of a decision tree can be explained using above binary tree. Let’s say you want to predict 
whether a person is fit given their information like age, eating habit, and physical activity, etc. The

the input is and what the corresponding output is in the training data) where the data is conti